# Projecte: BICITIERRA - Baseline

Ara se suposa que l'alumnat ja té clar els objectius del projecte i ha fet una primera exploració de les dades. En este quadern l'objectiu és crear baselines (models de referència, sense utilitzar algorismes de ML) per a la predicció de la variable objectiu. De moment, només en el dataset de vendes.

Els resultats de les prediccions que facen ací seran utilitzats com a referència per a comparar amb els resultats que s'obtinguen amb els models de Machine Learning.

## Ruta de treball

1. Triar variable objectiu.
2. Crear una baseline comprensible.
3. Comparar-la amb els valors reals.
4. Escriure què milloraries després.

In [ ]:
# En primer lloc anem a carregar les llibreries necessàries i llegir el dataset de vendes.
# tin en compte que la ruta el dataset por variar en funció de com tingues organitzada la carpeta del projecte
# mostrem les 10 primeres files per una primera inspecció visual

import pandas as pd

df = pd.read_csv('../data/originales/sales_history.csv')
df.head(10)

### Exemple: Baseline per mitjana de grup

Com a aproximació simple, predim `sales_units` amb la mitjana del grup `channel + product_line`.

In [ ]:
# calculem la mitjana mensual de vendes per cada combinació de `channel` i `product_line`
# dona-li un nom al resultat, per exemple "baseline_sales_units"
# i la guardem en una agrupació que podem anomenar `group_mean` 
df_grouped = df.groupby(['channel', 'product_line'])['sales_units'].mean().reset_index()
group_mean = df_grouped.rename(columns={'sales_units': 'baseline_sales_units'})

# investiga com adjuntar 2 dataframes utilitzant el mètode merge
# fes un merge per afegir la nova columna al dataframe original
# tenint en compte que en cada fila se posarà la mitjana corresponent al canal i el producte
# que corresponga
df = df.merge(group_mean, on=['channel', 'product_line'], how='left')

# ara podem mostrar el dataframe fent que apareguen el mes, el canal, el producte, les unitats
# venudes reals (sales_units) i la predicció (baseline_sales_units)
# mostra per exemple les primeres 15 files i fes una primera exploració visual
# per veure si hi ha molta diferència entre la predicció i la realitat
df[['month', 'channel', 'product_line', 'sales_units', 'baseline_sales_units']].head(15)

,month,channel,product_line,sales_units,baseline_sales_units
0,2021-01,store,urban,95,128.303030
1,2021-01,store,urban,82,128.303030
2,2021-01,store,urban,141,128.303030
3,2021-01,store,mountain,56,87.863636
4,2021-01,store,mountain,44,87.863636
5,2021-01,store,mountain,48,87.863636
6,2021-01,store,electric,39,55.454545
7,2021-01,store,electric,17,55.454545
8,2021-01,store,electric,48,55.454545
9,2021-01,ecommerce,urban,130,156.696970


In [38]:
# anem a veure la mitjana de l'error absolut (MAE) de la baseline
# l'error absolut es simplement el valor absolut de la diferencia entre el valor real i la predicció
# podeu crear una columna nova amb eixa informació
df['abs_error'] = (df['sales_units'] - df['baseline_sales_units']).abs()

# torneu a mostrar les primeres 15 files i que aparega també el MAE
df[['month', 'channel', 'product_line', 'sales_units', 'baseline_sales_units', 'abs_error']].head(15)

,month,channel,product_line,sales_units,baseline_sales_units,abs_error
0,2021-01,store,urban,95,128.303030,33.303030
1,2021-01,store,urban,82,128.303030,46.303030
2,2021-01,store,urban,141,128.303030,12.696970
3,2021-01,store,mountain,56,87.863636,31.863636
4,2021-01,store,mountain,44,87.863636,43.863636
5,2021-01,store,mountain,48,87.863636,39.863636
6,2021-01,store,electric,39,55.454545,16.454545
7,2021-01,store,electric,17,55.454545,38.454545
8,2021-01,store,electric,48,55.454545,7.454545
9,2021-01,ecommerce,urban,130,156.696970,26.696970


In [39]:
# podeu calcular l'error mig calculant la mitjana d'eixa columna nova que hem creat
# feu-ho i mostreu el resultat
mae = df['abs_error'].mean()
mae

np.float64(25.440507771315854)

In [40]:
# mostreu l'error en forma de percentatge sobre la totalitat de les vendes
# eixe valor guardeu-lo perquè després ens servirà per veure si la baseline
# millora o empitjora segons anem canviant coses
error_pct = 100 * df['abs_error'].sum() / df['sales_units'].sum()
error_pct

np.float64(25.378556748948892)

Eixe valor que acabeu de mostrar és el que vos ha de servir de referència amb la resta de baselines que feu, aplicant tots els canvis qeu se vos ocurrisquen. El repte és baixar eixe percentatge d'error.

In [104]:
# Per exemple, podem repetir tot el procés anterior provant a afegir la regió al grup i veure què passa.
group_mean_region = (
    df.groupby(['channel', 'product_line', 'region'])['sales_units']
      .mean()
      .reset_index()
      .rename(columns={'sales_units': 'baseline_region'})
)
df = df.merge(group_mean_region, on=['channel', 'product_line', 'region'], how='left')
df['abs_error_region'] = (df['sales_units'] - df['baseline_region']).abs()

mae_region = df['abs_error_region'].mean()
error_pct_region = 100 * df['abs_error_region'].sum() / df['sales_units'].sum()
print(f'MAE: {mae_region:.2f} | Error %: {error_pct_region:.2f} (abans: {mae:.2f} | {error_pct:.2f})')

MAE: 19.46 | Error %: 19.41 (abans: 25.44 | 25.38)


In [115]:
# Si no millora, treiem del grup la regió i afegim en el seu lloc la campanya 
# (columna campaign: ninguna, Nadal, Black Friday...)
group_mean_campaign = (
    df.groupby(['channel', 'product_line', 'campaign'])['sales_units']
      .mean()
      .reset_index()
      .rename(columns={'sales_units': 'baseline_campaign'})
)
# si la cel·la s'executa dos voltes, traiem la columna anterior perquè el merge no la duplique
df = df.drop(columns='baseline_campaign', errors='ignore')
df = df.merge(group_mean_campaign, on=['channel', 'product_line', 'campaign'], how='left')
df['abs_error_campaign'] = (df['sales_units'] - df['baseline_campaign']).abs()

mae_campaign = df['abs_error_campaign'].mean()
error_pct_campaign = 100 * df['abs_error_campaign'].sum() / df['sales_units'].sum()
print(f'MAE: {mae_campaign:.2f} | Error %: {error_pct_campaign:.2f}')

MAE: 23.65 | Error %: 23.59


In [125]:
# Una altra opció més complicada però que segurament donarà millors resultats és utilitzar
# la mitjana de les vendes només dels primers anys (per exemple 2021 a 2024) per a predir les vendes
# de 2025, i després comparar amb els valors reals (que els tenim)
train = df[df['month'].str[:4] <= '2024']
test = df[df['month'].str[:4] == '2025'].copy()

mean_train = (
    train.groupby(['channel', 'product_line', 'region'])['sales_units']
         .mean()
         .reset_index()
         .rename(columns={'sales_units': 'baseline_2021_2024'})
)
test = test.merge(mean_train, on=['channel', 'product_line', 'region'], how='left')
test['abs_error_2025'] = (test['sales_units'] - test['baseline_2021_2024']).abs()

mae_2025 = test['abs_error_2025'].mean()
error_pct_2025 = 100 * test['abs_error_2025'].sum() / test['sales_units'].sum()
print(f'MAE 2025: {mae_2025:.2f} | Error % 2025: {error_pct_2025:.2f}')

MAE 2025: 20.28 | Error % 2025: 19.54


### On falla més la baseline

Açò és útil perquè mostra si la simple agrupació ja captura prou bé el negoci o si encara queda molta variabilitat.

In [126]:
# Ordeneu per error absolut, de major a menor, i mostreu les 25 primeres files


In [127]:
# Ordeneu per percentatge d'error, de major a menor, i mostreu les 25 primeres files


En els llistats anteriors podreu veure coses com si els errors més grans se concentren en un canal, en un producte, en una regió, en un periode, etc.

In [128]:
# Com que ja sabem fer agrupacions, i ja tenim una columna nova amb l'error absolut
# podem agrupar per canal i mostrar l'error mig
#
# Creeu nous blocs de codi per a mostrar la mitjana de l'error seguint diferents agrupacions: 
# per producte, per regió, per producte i canal, per canal i campanya... Qualsevol combinació 
# que vos vinga al cap i que tinga sentit

In [129]:
# també podem comprovar si és cert que l'error sembla ser major en els mesos més recents
# per exemple, calculem l'error només a partir de 2025

In [130]:
# i ara l'anterior a 2025

In [131]:
# fins i tot, si finalment heu separat l'any i el mes en columnes diferents a month, podeu calcular
# la mitjana de l'error per any o per mes

#### Fer prediccions i testejar-les


Anem a fer prediccions com cal, testejant amb dades que no apareixen en l'entrenament de la baseline.

Per exemple, fem les prediccions amb les dades de 2021-2024, i utilitzem les de 2025 per validar la baseline.

Podeu fer-ho per qualsevol agrupació. En els comentaris següents la suggerència és per prediccions per famílies.

Seria convenient haver fet abans la separació de la columna month en una columna year_solt i una altra month_solt tal com hem vist a classe.

In [132]:
# 1. Vendes totals de cada família en cada mes

# 2. Separar l'historial disponible del període que volem predir
# 2.1 un dataframe per a les dades de 2021-2024
# 2.2 un altre dataframe per a les dades de 2025
# 3. Mitjana mensual històrica de cada família en el dataframe de 2021-2024. Guardem el resultat
# en una columna nova
# 4. Afegim eixa columna nova (els mesos de prova) al dataframe de 2025
# 5. Mostrem un llistat del dataframe de 2025 (per exemple les primeres 25 files) on aparega:
# - l'any (serà 2025)
# - el mes (columna month_solt)
# - la família (product_line)
# - les vendes reals (sales_units)
# - la predicció de vendes per a eixa família

In [133]:
# calculem la mitjana de l'error absolut (valor absolut de la diferència entre valor real i predicció)

# el mostrem

# si el valor és alt pot ser que el model funciona mal, però també pot ser que els valors de les vendes 
# siguen alts

# per això és millor treure l'error relatiu 
# (recordeu que estem en el dataframe de 2025)
# ho podem fer simplement sumant la columna sales_units per una banda
# sumant d'una altra banda els errors absoluts (valor absolut de vendes reals - predicció)
# calculant quin percentatge suposa la suma dels errors absoluts sobre el total de vendes
# (una senzilla regla de tres: 100 * suma_errors / suma_vendes)


Hem de tenir en compte que estem calculant errors absoluts. És a dir, la desviació de la predicció respecte a la quantitat real. Això implica que quan parlem de quantitats grans, la desviació absoluta també serà més gran. Per exemple, pot semblar que una desviació de 100 unitats és major que una de 5, però si la desviació de 100 unitats és respecte a quantitats de cinc xifres sempre serà molt millor que una de 5 respecte a quantitats de 2 xifres. Per això una de les coses que podríem fer a partir d'ací és tornar a fer tots els càlculs però en lloc de mostrar l'error en valor absolut, fer-ho amb el percentatge respecte al valor real, per exemple.

A continuació hi ha un exemple si cal explicar com fer-ho, després podem deixar que l'alumnat revise totes les proves que ha fet mostrant els errors en valor absolut i també en percentatge.

Podeu repetir l'exercici tantes vegades com tingueu ganes, amb agrupacions i periodes diferents. Per exemple, utilitzant la mitjana només de 2022-2024 en lloc de 2021-2024, o agrupar per familia i mes, per campanya...